# Stock Performance Prediction — Model 1: Bidirectional GRU

**Project:** Stock Performance Prediction (RNN)  
**Prerequisites:** `01_exploratory_data_analysis.ipynb`, `02_preprocessing.ipynb`  
**Loads from:** `data/processed/aapl_sequences.npz`, `scaler.json`, `pipeline_config.json`

This notebook trains and evaluates a **Stacked Bidirectional GRU** classifier for next-day direction prediction on AAPL.

### Why Bidirectional GRU?
- **GRU over LSTM**: Fewer parameters, less overfitting risk on ~6,200 training samples. Empirically matches LSTM on short financial sequences (60-day lookback).
- **Bidirectional**: Both forward and backward passes over the lookback window — the model can learn that a pattern at day 10 of the window is more significant given what happened at day 40. No data leakage: the full 60-day window is historical at prediction time.
- **Stacked (2 layers)**: Layer 1 learns local short-term patterns (momentum, day-to-day returns); Layer 2 combines these into regime-level signals (volatility clustering, trend persistence).

### Pipeline
```
Load artifacts → Build model → Train with early stopping
→ Evaluate (Val + Test) → Baselines → Save model
```

| Stage | Section |
|-------|---------|
| Setup & load | 1–2 |
| Model definition | 3 |
| Training | 4 |
| Evaluation | 5 |
| Baselines | 6 |
| Save | 7 |

## 1. Setup & Dependencies

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED = ["numpy", "pandas", "matplotlib", "seaborn", "scikit-learn", "tensorflow"]

def ensure_packages(packages):
    missing = [p for p in packages if importlib.util.find_spec(p.replace("-", "_")) is None]
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

ensure_packages(REQUIRED)

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, accuracy_score, roc_curve
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams["figure.dpi"] = 110

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow: {tf.__version__}")
print(f"Python:     {sys.executable}")
print(f"GPU available: {bool(tf.config.list_physical_devices('GPU'))}")

## 2. Load Preprocessed Artifacts

In [ ]:
def find_project_root() -> Path:
    candidates = [Path.cwd()]
    if Path.cwd().name == "notebooks":
        candidates.append(Path.cwd().parent)
    candidates.extend(Path.cwd().parents)
    for path in candidates:
        if (path / "Stock Market Dataset").exists():
            return path
    return Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


PROJECT_ROOT  = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR    = PROJECT_ROOT / "data" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root:  {PROJECT_ROOT}")
print(f"Processed dir: {PROCESSED_DIR}")
print(f"Models dir:    {MODELS_DIR}")

In [ ]:
# ── Load sequence arrays ──────────────────────────────────────────────────
seq_path = PROCESSED_DIR / "aapl_sequences.npz"
if not seq_path.exists():
    raise FileNotFoundError(
        f"Sequences not found at {seq_path}.\n"
        "Run 02_preprocessing.ipynb first."
    )

data = np.load(seq_path, allow_pickle=True)
X_train, y_train = data["X_train"], data["y_train"]
X_val,   y_val   = data["X_val"],   data["y_val"]
X_test,  y_test  = data["X_test"],  data["y_test"]
feature_cols     = list(data["feature_cols"])

print("Sequence shapes loaded:")
for split, X, y in [("train", X_train, y_train),
                    ("val",   X_val,   y_val),
                    ("test",  X_test,  y_test)]:
    up_pct = y.mean() * 100
    print(f"  {split:<6}  X={X.shape}  y={y.shape}  up%={up_pct:.1f}")

print(f"\nFeatures ({len(feature_cols)}): {feature_cols}")

In [ ]:
# ── Load config & scaler ──────────────────────────────────────────────────
with open(PROCESSED_DIR / "pipeline_config.json") as f:
    cfg = json.load(f)

with open(PROCESSED_DIR / "scaler.json") as f:
    scaler = json.load(f)

LOOKBACK   = cfg["lookback_days"]   # 60
N_FEATURES = len(feature_cols)      # 7

print(f"Lookback: {LOOKBACK} days | Features: {N_FEATURES}")
print(f"Train end: {cfg['train_end']}  |  Val: {cfg['val_start']} – {cfg['val_end']}  |  Test: {cfg['test_start']} – {cfg['test_end']}")

## 3. Model Definition — Stacked Bidirectional GRU

### Architecture
```
Input (60, 7)
  └─ Bidirectional GRU (64 units, return_sequences=True) + Dropout(0.3)
       └─ Bidirectional GRU (32 units, return_sequences=False) + Dropout(0.3)
            └─ Dense (32, ReLU) + L2 regularisation
                 └─ Dropout(0.2)
                      └─ Dense (1, Sigmoid)  →  P(next day UP)
```

**Design choices:**
- Units halved in the second GRU layer (64 → 32) to create a compression bottleneck that forces the model to learn a compact representation.
- `return_sequences=True` on layer 1 passes the full sequence to layer 2; `False` on layer 2 collapses the temporal axis before the dense head.
- Dropout after each recurrent layer to regularise the ~6,200-sample training set.
- L2 on the dense layer as a secondary regulariser.
- Single sigmoid output: probability of next-day UP move.

In [ ]:
def build_bigru(lookback: int, n_features: int, seed: int = 42) -> keras.Model:
    """Stacked Bidirectional GRU for binary direction classification."""
    tf.random.set_seed(seed)

    inputs = keras.Input(shape=(lookback, n_features), name="sequence_input")

    # ── Layer 1: Bidirectional GRU (returns full sequence) ────────────────
    x = layers.Bidirectional(
        layers.GRU(64, return_sequences=True, dropout=0.0,
                   recurrent_dropout=0.0, name="gru_1"),
        name="bigru_1"
    )(inputs)
    x = layers.Dropout(0.3, seed=seed, name="drop_1")(x)

    # ── Layer 2: Bidirectional GRU (collapses sequence) ───────────────────
    x = layers.Bidirectional(
        layers.GRU(32, return_sequences=False, dropout=0.0,
                   recurrent_dropout=0.0, name="gru_2"),
        name="bigru_2"
    )(x)
    x = layers.Dropout(0.3, seed=seed, name="drop_2")(x)

    # ── Dense head ────────────────────────────────────────────────────────
    x = layers.Dense(
        32, activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
        name="dense_1"
    )(x)
    x = layers.Dropout(0.2, seed=seed, name="drop_3")(x)

    outputs = layers.Dense(1, activation="sigmoid", name="output")(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name="BiGRU_DirectionClassifier")
    return model


model = build_bigru(LOOKBACK, N_FEATURES, seed=SEED)
model.summary(line_length=80)

In [ ]:
# ── Compile ───────────────────────────────────────────────────────────────
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.AUC(name="auc"),
    ]
)
print("Model compiled.")
print(f"Total trainable parameters: {model.count_params():,}")

## 4. Training

**Callbacks:**
- `EarlyStopping` — monitors `val_auc`; stops if no improvement for 15 epochs; restores the best weights automatically.
- `ReduceLROnPlateau` — halves the learning rate if `val_loss` stalls for 7 epochs (floor: 1e-5).
- `ModelCheckpoint` — saves the best model to disk by `val_auc`.

**Note on COVID test set:** The test set (2020 Q1) is a held-out stress window. It is never seen during training or used for early stopping decisions.

In [ ]:
CHECKPOINT_PATH = MODELS_DIR / "bigru_best.keras"

cb_early = callbacks.EarlyStopping(
    monitor="val_auc",
    patience=15,
    mode="max",
    restore_best_weights=True,
    verbose=1,
)
cb_reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=7,
    min_lr=1e-5,
    verbose=1,
)
cb_checkpoint = callbacks.ModelCheckpoint(
    filepath=str(CHECKPOINT_PATH),
    monitor="val_auc",
    mode="max",
    save_best_only=True,
    verbose=0,
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    callbacks=[cb_early, cb_reduce_lr, cb_checkpoint],
    verbose=1,
    shuffle=False,  # preserve time order within each epoch
)

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────
hist = pd.DataFrame(history.history)
epochs_ran = len(hist)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (train_col, val_col, title) in zip(axes, [
    ("loss",     "val_loss",     "Binary Cross-Entropy Loss"),
    ("accuracy", "val_accuracy", "Accuracy"),
    ("auc",      "val_auc",      "AUC-ROC"),
]):
    ax.plot(hist[train_col], label="Train", linewidth=1.8)
    ax.plot(hist[val_col],   label="Val",   linewidth=1.8, linestyle="--")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.axvline(hist[val_col].idxmax() if "auc" in val_col else hist[val_col].idxmin(),
               color="gray", linewidth=0.8, linestyle=":")

fig.suptitle(f"BiGRU — Training History ({epochs_ran} epochs)", y=1.02)
plt.tight_layout()
plt.show()

best_epoch = hist["val_auc"].idxmax()
print(f"Best epoch: {best_epoch + 1}  |  val_auc={hist['val_auc'][best_epoch]:.4f}  |  val_loss={hist['val_loss'][best_epoch]:.4f}")

## 5. Evaluation

Evaluation is run on both the **validation set** (2018–2019, normal market) and the **test set** (2020 Q1, COVID stress window) separately. Results are not aggregated — the two regimes represent fundamentally different market conditions.

**Primary metrics:** F1-score, AUC-ROC  
**Secondary metrics:** Accuracy, Precision, Recall  
**Why not just accuracy:** The direction target is ~50/50 balanced, so a naive 'always predict UP' model achieves ~50% accuracy. F1 and AUC better reflect real predictive value.

In [ ]:
def evaluate_split(model, X, y_true, split_name: str, threshold: float = 0.5) -> dict:
    """Full evaluation on a single split. Returns a metrics dict."""
    y_prob  = model.predict(X, verbose=0).flatten()
    y_pred  = (y_prob >= threshold).astype(int)

    acc     = accuracy_score(y_true, y_pred)
    f1      = f1_score(y_true, y_pred, zero_division=0)
    auc     = roc_auc_score(y_true, y_prob)

    print(f"\n{'='*55}")
    print(f"  {split_name.upper()} RESULTS  (n={len(y_true)}, threshold={threshold})")
    print(f"{'='*55}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  F1-score : {f1:.4f}")
    print(f"  AUC-ROC  : {auc:.4f}")
    print(f"\n{classification_report(y_true, y_pred, target_names=['DOWN (0)', 'UP (1)'], zero_division=0)}")

    return {"split": split_name, "accuracy": acc, "f1": f1, "auc": auc,
            "y_prob": y_prob, "y_pred": y_pred, "y_true": y_true}


val_results  = evaluate_split(model, X_val,  y_val,  "Validation (2018–2019)")
test_results = evaluate_split(model, X_test, y_test, "Test — COVID Stress (2020 Q1)")

In [ ]:
# ── Confusion matrices ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, res in zip(axes, [val_results, test_results]):
    cm = confusion_matrix(res["y_true"], res["y_pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["DOWN", "UP"], yticklabels=["DOWN", "UP"])
    ax.set_title(f"{res['split']}\nF1={res['f1']:.3f}  AUC={res['auc']:.3f}")
    ax.set_ylabel("Actual")
    ax.set_xlabel("Predicted")

fig.suptitle("BiGRU — Confusion Matrices", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── ROC curves ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))

for res, color in zip([val_results, test_results], ["steelblue", "crimson"]):
    fpr, tpr, _ = roc_curve(res["y_true"], res["y_prob"])
    ax.plot(fpr, tpr, label=f"{res['split']} (AUC={res['auc']:.3f})",
            color=color, linewidth=2)

ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, label="Random (AUC=0.500)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("BiGRU — ROC Curves")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Predicted probability distributions ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, res in zip(axes, [val_results, test_results]):
    for label, color, name in [(0, "#c0392b", "DOWN"), (1, "#27ae60", "UP")]:
        mask = res["y_true"] == label
        ax.hist(res["y_prob"][mask], bins=30, alpha=0.6,
                color=color, label=f"Actual {name}")
    ax.axvline(0.5, color="black", linewidth=1, linestyle="--", label="Threshold=0.5")
    ax.set_xlabel("Predicted P(UP)")
    ax.set_ylabel("Count")
    ax.set_title(f"{res['split']}")
    ax.legend(fontsize=8)

fig.suptitle("BiGRU — Predicted Probability Distributions by True Class", y=1.02)
plt.tight_layout()
plt.show()

## 6. Baseline Comparisons

A model that does not beat simple baselines provides no evidence of sequence learning.

| Baseline | Description |
|----------|-------------|
| Always UP | Predicts 1 (up) for every sample |
| Always DOWN | Predicts 0 (down) for every sample |
| Last direction | Predicts same direction as the most recent day in the lookback window |

In [ ]:
def baseline_metrics(y_true, y_pred, name: str) -> dict:
    return {
        "Model": name,
        "Accuracy": round(accuracy_score(y_true, y_pred), 4),
        "F1": round(f1_score(y_true, y_pred, zero_division=0), 4),
    }


rows = []
for split_name, X, y in [("Validation", X_val, y_val), ("Test", X_test, y_test)]:
    # Last log_return in each sequence → direction proxy (feature index 0 = log_return)
    last_return = X[:, -1, 0]  # most recent day's log_return
    last_dir    = (last_return > 0).astype(int)

    bigru_pred = (model.predict(X, verbose=0).flatten() >= 0.5).astype(int)

    for row in [
        baseline_metrics(y, np.ones_like(y, dtype=int),  f"[{split_name}] Always UP"),
        baseline_metrics(y, np.zeros_like(y, dtype=int), f"[{split_name}] Always DOWN"),
        baseline_metrics(y, last_dir,                    f"[{split_name}] Last Direction"),
        {"Model": f"[{split_name}] BiGRU (ours)",
         "Accuracy": round(accuracy_score(y, bigru_pred), 4),
         "F1":       round(f1_score(y, bigru_pred, zero_division=0), 4)},
    ]:
        rows.append(row)

baseline_df = pd.DataFrame(rows)
print("\nBaseline comparison:")
display(baseline_df.set_index("Model"))

In [ ]:
# ── Bar chart: F1 comparison ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
colors_bar = ["#bdc3c7"] * (len(baseline_df) - 2) + ["#2980b9", "#2980b9"]
colors_bar = [
    "#2980b9" if "BiGRU" in m else "#bdc3c7"
    for m in baseline_df["Model"]
]
ax.barh(baseline_df["Model"], baseline_df["F1"], color=colors_bar, edgecolor="white")
ax.axvline(0.5, color="gray", linewidth=0.8, linestyle="--")
ax.set_xlabel("F1-Score")
ax.set_title("BiGRU vs Baselines — F1-Score")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Save Model & Results

In [ ]:
# ── Save final model ──────────────────────────────────────────────────────
final_path = MODELS_DIR / "bigru_final.keras"
model.save(final_path)
print(f"Model saved: {final_path}")

# ── Save predictions for ensemble notebook ────────────────────────────────
preds = {
    "val_prob":  val_results["y_prob"].tolist(),
    "val_pred":  val_results["y_pred"].tolist(),
    "val_true":  val_results["y_true"].tolist(),
    "test_prob": test_results["y_prob"].tolist(),
    "test_pred": test_results["y_pred"].tolist(),
    "test_true": test_results["y_true"].tolist(),
    "metrics": {
        "val":  {"accuracy": val_results["accuracy"],  "f1": val_results["f1"],  "auc": val_results["auc"]},
        "test": {"accuracy": test_results["accuracy"], "f1": test_results["f1"], "auc": test_results["auc"]},
    }
}
preds_path = MODELS_DIR / "bigru_predictions.json"
with open(preds_path, "w") as f:
    json.dump(preds, f, indent=2)
print(f"Predictions saved: {preds_path}")

# ── Summary table ─────────────────────────────────────────────────────────
summary = pd.DataFrame([
    {"Split": "Validation (2018–2019)",      **preds["metrics"]["val"]},
    {"Split": "Test — COVID Stress (2020 Q1)", **preds["metrics"]["test"]},
])
print("\n── BiGRU Final Results ──")
display(summary.set_index("Split").round(4))

---

## Summary

This notebook trained and evaluated a **Stacked Bidirectional GRU** (2 layers: 64 → 32 units, both bidirectional) on AAPL next-day direction prediction using 60-day sequences of 7 engineered features.

**Key artifacts produced:**
- `data/models/bigru_best.keras` — best checkpoint (by val AUC)
- `data/models/bigru_final.keras` — final model
- `data/models/bigru_predictions.json` — probabilities and metrics for ensemble use

**Next:** `03b_model_tcn.ipynb` — Temporal Convolutional Network (second ensemble member).